# AI Prep Search via Foundation Models

This notebook builds and deploys an MLflow pyfunc model that recreates the `ai_prep_search` functionality using Databricks Foundation Model APIs. It accepts the structured JSON output from `ai_parse_document` (or plain text/markdown), performs semantic chunking, enriches each chunk with document context via an LLM, and produces output compatible with Databricks AI Search indexes.

The foundation model is configurable via the `CHUNKING_MODEL` variable below.

In [0]:
# No heavy ML packages — the foundation model is called remotely via FMAPI.
%pip install -U \
  "mlflow>=3.6.0" \
  "openai>=1.37.0,<4.0.0"
%restart_python

In [0]:
# --- Workspace configuration ---
# Update CATALOG and SCHEMA to match your Unity Catalog location.
CATALOG = "justinm_demo"
SCHEMA = "parse"

# Foundation model for chunking enrichment (context sentence, table summaries,
# related questions). Change this to any FMAPI-compatible model.
CHUNKING_MODEL = "databricks-gemini-3-8-flash"

REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.ai-prep-search-fmapi"
ENDPOINT_NAME = "ai-prep-search-fmapi"

# Chunking parameters
MAX_CHUNK_CHARS = 2000   # Approximate max characters per chunk
LLM_MAX_TOKENS = 2048    # Max tokens for enrichment LLM calls

print({
    "chunking_model": CHUNKING_MODEL,
    "registered_model": REGISTERED_MODEL_NAME,
    "endpoint": ENDPOINT_NAME,
})

In [0]:
import json
import time
import uuid

import mlflow
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from mlflow import MlflowClient
from mlflow.models import ModelSignature
from mlflow.models.resources import DatabricksServingEndpoint
from mlflow.types.schema import ColSpec, DataType, Schema

mlflow.set_registry_uri("databricks-uc")
print("Imports loaded")

In [0]:
class AiPrepSearchModel(mlflow.pyfunc.PythonModel):
    """Recreates ai_prep_search using Databricks Foundation Model APIs.

    Accepts structured ai_parse_document 2.0 JSON or plain text/markdown,
    splits into semantic chunks, enriches with document context via an LLM,
    and returns search-ready output matching the ai_prep_search schema.
    """

    _CONTEXT_PROMPT = (
        "You are a document analysis assistant. Given the following document "
        "text, write a single sentence that describes the document's main "
        "topic and purpose. This sentence will provide context for search "
        "indexing. Return ONLY the context sentence, nothing else.\n\n"
        "Document text (first 3000 characters):\n{text}"
    )

    _TABLE_SUMMARY_PROMPT = (
        "You are a data analysis assistant. Given the following table from a "
        "document, provide:\n"
        "1. A brief summary of what the table contains (1-2 sentences)\n"
        "2. 3-5 natural language questions this table could answer\n\n"
        "Return ONLY a JSON object: "
        '{{"summary": "...", "questions": ["...", "..."]}}\n\n'
        "Table content:\n{table_content}"
    )

    def __init__(self, chunking_model=CHUNKING_MODEL,
                 max_chunk_chars=MAX_CHUNK_CHARS,
                 llm_max_tokens=LLM_MAX_TOKENS):
        self.chunking_model = chunking_model
        self.max_chunk_chars = max_chunk_chars
        self.llm_max_tokens = llm_max_tokens

    def load_context(self, context):
        import logging
        self.log = logging.getLogger("ai_prep_search_fmapi")
        self._ws = WorkspaceClient()
        self.client = openai.OpenAI(
            api_key=self._get_token(),
            base_url=f"{self._ws.config.host}/serving-endpoints",
        )

    def _get_token(self):
        """Fetch a fresh bearer token. In Model Serving, DATABRICKS_TOKEN is
        auto-rotated by the serving infra; fall back to WorkspaceClient auth
        in notebook context."""
        import os
        token = os.environ.get("DATABRICKS_TOKEN") or self._ws.config.token
        if not token:
            headers = self._ws.config.authenticate()
            token = headers.get("Authorization", "").removeprefix("Bearer ").strip()
        return token

    # ── LLM helpers ──────────────────────────────────────────────────

    def _llm_call(self, prompt, max_tokens=None):
        """Single LLM call; returns text or empty string on failure."""
        try:
            resp = self.client.chat.completions.create(
                model=self.chunking_model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens or self.llm_max_tokens,
            )
            content = resp.choices[0].message.content
            if isinstance(content, list):
                return "".join(
                    p.get("text", "") if isinstance(p, dict)
                    else getattr(p, "text", "")
                    for p in content
                ).strip()
            return (content or "").strip()
        except Exception as exc:
            self.log.warning("LLM call failed: %s", exc)
            if not hasattr(self, "_llm_errors"):
                self._llm_errors = []
            self._llm_errors.append(f"llm_call_failed: {type(exc).__name__}: {exc}"[:500])
            return ""

    def _generate_context_sentence(self, full_text):
        """Generate a one-line document context sentence."""
        return self._llm_call(
            self._CONTEXT_PROMPT.format(text=full_text[:3000]),
            max_tokens=256,
        )

    def _generate_table_enrichment(self, table_html):
        """Generate table summary and related questions."""
        raw = self._llm_call(
            self._TABLE_SUMMARY_PROMPT.format(table_content=table_html[:3000]),
            max_tokens=512,
        )
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
            if raw.endswith("```"):
                raw = raw[:-3].strip()
        try:
            result = json.loads(raw)
            return result.get("summary", ""), result.get("questions", [])
        except (json.JSONDecodeError, AttributeError):
            return "", []

    _METADATA_ENRICHMENT_PROMPT = (
        "You are a document analysis assistant. Given the following document "
        "content, extract or infer metadata for search indexing.\n\n"
        "Document text (first 4000 characters):\n{text}\n\n"
        "Source file: {source_uri}\n"
        "Number of content sections: {n_chunks}\n\n"
        "Return a JSON object with these fields:\n"
        '{{\n'
        '  "document_title": "The document\'s main title or a concise descriptive name",\n'
        '  "sections": ["A short descriptive section/topic label for each of the {n_chunks} content sections, in document order"],\n'
        '  "page_header_default": "A reasonable default page header (e.g., organization name or document title abbreviation)",\n'
        '  "page_footer_default": "A reasonable default page footer (e.g., source, date, or document classification)"\n'
        '}}\n\n'
        "Return ONLY the JSON object."
    )

    def _generate_metadata_enrichment(self, full_text, source_uri, n_chunks, doc_ctx):
        """Generate enriched metadata for the document in a single LLM call."""
        has_title = bool(doc_ctx.get("title", "").strip())
        has_sections = len(doc_ctx.get("section_headers", [])) >= max(1, n_chunks // 2)
        has_headers = bool(doc_ctx.get("page_headers"))
        has_footers = bool(doc_ctx.get("page_footers"))

        if has_title and has_sections and has_headers and has_footers:
            return {
                "document_title": doc_ctx["title"],
                "sections": [sh["text"] for sh in doc_ctx["section_headers"]],
                "page_header_default": doc_ctx["title"],
                "page_footer_default": next(iter(doc_ctx["page_footers"].values()), ""),
            }

        raw = self._llm_call(
            self._METADATA_ENRICHMENT_PROMPT.format(
                text=full_text[:4000],
                source_uri=source_uri or "(unknown)",
                n_chunks=n_chunks,
            ),
            max_tokens=512,
        )
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
            if raw.endswith("```"):
                raw = raw[:-3].strip()
        try:
            result = json.loads(raw)
        except (json.JSONDecodeError, AttributeError):
            result = {}

        title = doc_ctx.get("title") or result.get("document_title", "")
        filename = source_uri.rsplit("/", 1)[-1] if source_uri else ""
        return {
            "document_title": title,
            "sections": result.get("sections", []),
            "page_header_default": result.get("page_header_default", title),
            "page_footer_default": result.get("page_footer_default", filename),
        }

    # ── Input parsing ────────────────────────────────────────────────

    def _parse_input(self, parsed_doc_json):
        """Detect whether input is ai_parse_document 2.0 JSON or plain text.

        Returns (elements_or_text, pages, metadata_dict, is_plain_text).
        """
        if not parsed_doc_json or (
            isinstance(parsed_doc_json, float) and pd.isna(parsed_doc_json)
        ):
            return "", None, None, True
        text = str(parsed_doc_json).strip()
        try:
            data = json.loads(text)
            if isinstance(data, dict) and "document" in data:
                doc = data["document"]
                return (
                    doc.get("elements", []),
                    doc.get("pages", []),
                    data.get("metadata", {}),
                    False,
                )
        except (json.JSONDecodeError, TypeError):
            pass
        return text, None, None, True

    # ── Document context extraction ──────────────────────────────────

    def _extract_document_context(self, elements):
        """Pull document-level metadata from typed elements."""
        ctx = {
            "title": "",
            "section_headers": [],
            "page_headers": {},
            "page_footers": {},
            "page_numbers": {},
            "captions": [],
            "footnotes": [],
        }
        for el in elements:
            t = el.get("type", "text")
            c = el.get("content") or ""
            pids = [b.get("page_id", 0) for b in (el.get("bbox") or [])]
            pid = pids[0] if pids else 0
            eid = el.get("id", 0)
            if t == "title" and not ctx["title"]:
                ctx["title"] = c
            elif t == "section_header":
                ctx["section_headers"].append({"text": c, "element_id": eid})
            elif t == "page_header":
                ctx["page_headers"][pid] = c
            elif t == "page_footer":
                ctx["page_footers"][pid] = c
            elif t == "page_number":
                ctx["page_numbers"][pid] = c
            elif t == "caption":
                ctx["captions"].append({"text": c, "page_id": pid})
            elif t == "footnote":
                ctx["footnotes"].append({"text": c, "page_id": pid})
        return ctx

    # ── Semantic chunking ────────────────────────────────────────────

    def _semantic_chunk_elements(self, elements):
        """Group elements into semantic chunks by type and size."""
        BOUNDARY = {"title", "section_header"}
        STANDALONE = {"table", "figure"}
        SKIP = {"page_header", "page_footer", "page_number"}

        chunks, cur_els, cur_len = [], [], 0

        def flush():
            nonlocal cur_els, cur_len
            if cur_els:
                chunks.append(cur_els[:])
            cur_els, cur_len = [], 0

        for el in elements:
            t = el.get("type", "text")
            if t in SKIP:
                continue
            c = el.get("content") or el.get("description") or ""

            if t in STANDALONE:
                flush()
                chunks.append([el])
                continue
            if t in BOUNDARY and cur_els:
                flush()
            if cur_len and cur_len + len(c) > self.max_chunk_chars:
                flush()

            cur_els.append(el)
            cur_len += len(c)

        flush()
        return chunks

    def _chunk_plain_text(self, text):
        """Chunk plain text/markdown by semantic boundaries respecting size.

        Detects markdown headers (#, ##, etc.) and HTML tables as chunk
        boundaries, in addition to paragraph splitting for size limits.
        Mirrors the structured path's element-type-based boundaries so free
        text produces equally granular, semantically coherent chunks.
        """
        import re

        if not text.strip():
            return [""]

        # Split at markdown headers and HTML table blocks, keeping each
        # header with the content that follows it until the next boundary.
        boundary_re = re.compile(
            r'^(#{1,6}\s+.+?)$'             # markdown headers (non-greedy w/ DOTALL)
            r'|<table[^>]*>.*?</table>',     # HTML tables
            re.MULTILINE | re.IGNORECASE | re.DOTALL,
        )
        parts, pos = [], 0
        for m in boundary_re.finditer(text):
            if m.start() > pos:
                parts.append(text[pos:m.start()])
            parts.append(text[m.start():m.end()])
            pos = m.end()
        if pos < len(text):
            parts.append(text[pos:])

        chunks, cur = [], ""
        for part in parts:
            part = part.strip()
            if not part:
                continue

            is_html_table = part.lower().startswith("<table")
            is_header = bool(re.match(r"^#{1,6}\s+", part))

            # HTML tables are standalone chunks (like structured elements)
            if is_html_table:
                if cur:
                    chunks.append(cur)
                    cur = ""
                chunks.append(part)
                continue

            # Markdown headers start a new chunk (section boundary)
            if is_header and cur:
                chunks.append(cur)
                cur = ""

            # Split into paragraphs for size management
            for para in [p.strip() for p in part.split("\n\n") if p.strip()]:
                if cur and len(cur) + len(para) + 2 > self.max_chunk_chars:
                    chunks.append(cur)
                    cur = para
                else:
                    cur = f"{cur}\n\n{para}" if cur else para

        if cur:
            chunks.append(cur)
        return chunks if chunks else [text[:self.max_chunk_chars]]

    # ── chunk_to_embed construction ──────────────────────────────────

    def _get_chunk_context(self, chunk_elements, doc_ctx):
        """Resolve nearest page header/footer/section/caption/footnote."""
        page_ids = set()
        for el in chunk_elements:
            for b in (el.get("bbox") or []):
                page_ids.add(b.get("page_id", 0))
        primary = min(page_ids) if page_ids else 0

        min_eid = min(
            (el.get("id", float("inf")) for el in chunk_elements),
            default=float("inf"),
        )
        section = ""
        for sh in doc_ctx.get("section_headers", []):
            if sh["element_id"] <= min_eid:
                section = sh["text"]

        caption, footnote = "", ""
        for cap in doc_ctx.get("captions", []):
            if cap["page_id"] in page_ids:
                caption = cap["text"]
        for fn in doc_ctx.get("footnotes", []):
            if fn["page_id"] in page_ids:
                footnote = fn["text"]

        return {
            "page_header": doc_ctx.get("page_headers", {}).get(primary, ""),
            "page_footer": doc_ctx.get("page_footers", {}).get(primary, ""),
            "section_header": section,
            "caption": caption,
            "footnote": footnote,
            "page_number": str(doc_ctx.get("page_numbers", {}).get(primary, "")) or str(primary + 1),
            "page_ids": sorted(page_ids),
        }

    @staticmethod
    def _build_chunk_to_embed(chunk_text, doc_ctx, chunk_ctx,
                               is_table=False, table_summary="",
                               related_questions=None):
        """Assemble the enriched chunk_to_embed following the ai_prep_search
        template: fixed metadata labels are always rendered; optional items
        (context sentence, table summary, related questions) appear only when
        available."""
        fields = [
            ("Document Title", doc_ctx.get('title', '')),
            ("Page Header", chunk_ctx.get('page_header', '')),
            ("Page Footer", chunk_ctx.get('page_footer', '')),
            ("Section Header", chunk_ctx.get('section_header', '')),
            ("Caption", chunk_ctx.get('caption', '')),
            ("Footnote", chunk_ctx.get('footnote', '')),
            ("Page Number", chunk_ctx.get('page_number', '')),
        ]
        lines = [f"{label}: {value}" for label, value in fields if value]
        ctx_sentence = doc_ctx.get("context_sentence", "")
        if ctx_sentence:
            lines.append(ctx_sentence)
        if is_table and table_summary:
            lines.append(f"Table summary: {table_summary}")
        lines.append(f"Content:\n{chunk_text}")
        if is_table and related_questions:
            lines.append(
                "Related questions:\n"
                + "\n".join(f"- {q}" for q in related_questions)
            )
        return "\n".join(lines)

    # ── Row handlers ─────────────────────────────────────────────────

    def _handle_structured(self, elements, pages, source_uri, doc_id):
        """Process ai_parse_document 2.0 structured input."""
        doc_ctx = self._extract_document_context(elements)

        full_text = "\n".join(
            el.get("content") or el.get("description") or ""
            for el in elements
            if el.get("type") not in {"page_header", "page_footer", "page_number"}
        )
        doc_ctx["context_sentence"] = (
            self._generate_context_sentence(full_text) if full_text.strip() else ""
        )

        element_chunks = self._semantic_chunk_elements(elements)

        # Enrich missing document metadata with a single LLM call
        enriched = self._generate_metadata_enrichment(
            full_text, source_uri, len(element_chunks), doc_ctx
        )
        if not doc_ctx["title"]:
            doc_ctx["title"] = enriched.get("document_title", "")

        output_chunks = []
        for pos, chunk_els in enumerate(element_chunks):
            texts, is_table, is_figure = [], False, False
            for el in chunk_els:
                t = el.get("type", "text")
                if t == "table":
                    is_table = True
                    texts.append(el.get("content") or "")
                elif t == "figure":
                    is_figure = True
                    texts.append(f'[Figure: {el.get("description") or ""}]')
                else:
                    texts.append(el.get("content") or "")

            chunk_to_retrieve = "\n".join(texts)
            if not chunk_to_retrieve.strip():
                continue

            chunk_ctx = self._get_chunk_context(chunk_els, doc_ctx)

            # ── Fill missing fields from LLM-enriched metadata ───────
            sections = enriched.get("sections", [])
            if not chunk_ctx["section_header"] and pos < len(sections):
                chunk_ctx["section_header"] = sections[pos]
            if not chunk_ctx["page_header"]:
                chunk_ctx["page_header"] = enriched.get(
                    "page_header_default", doc_ctx.get("title", "")
                )
            if not chunk_ctx["page_footer"]:
                chunk_ctx["page_footer"] = enriched.get("page_footer_default", "")

            table_summary, related_qs = "", []
            if is_table:
                table_summary, related_qs = self._generate_table_enrichment(
                    chunk_to_retrieve
                )
                if not chunk_ctx["caption"] and table_summary:
                    chunk_ctx["caption"] = table_summary
            if is_figure and not chunk_ctx["caption"]:
                fig_descs = [
                    el.get("description", "") for el in chunk_els
                    if el.get("type") == "figure" and el.get("description")
                ]
                if fig_descs:
                    chunk_ctx["caption"] = fig_descs[0]

            chunk_to_embed = self._build_chunk_to_embed(
                chunk_to_retrieve, doc_ctx, chunk_ctx,
                is_table=is_table, table_summary=table_summary,
                related_questions=related_qs,
            )

            chunk_pages = []
            for pid in chunk_ctx["page_ids"]:
                ref = {"page_id": pid, "image_uri": ""}
                for p in (pages or []):
                    if p.get("id") == pid:
                        ref["image_uri"] = p.get("image_uri", "")
                        break
                chunk_pages.append(ref)

            output_chunks.append({
                "chunk_id": f"{doc_id}_{pos}",
                "chunk_position": pos,
                "chunk_to_retrieve": chunk_to_retrieve,
                "chunk_to_embed": chunk_to_embed,
                "metadata": {},
                "pages": chunk_pages,
            })

        return {
            "document": {
                "contents": output_chunks,
                "pages": pages or [],
                "source_uri": source_uri,
            },
            "error_status": [],
        }

    def _handle_plain_text(self, text, source_uri, doc_id):
        """Process plain text or markdown input with full enrichment.

        Applies the same LLM-driven metadata enrichment (document title,
        section labels, page header/footer defaults) and table enrichment
        as the structured path, so free-text input produces equally rich
        chunk_to_embed output.
        """
        import re

        ctx_sentence = (
            self._generate_context_sentence(text) if text.strip() else ""
        )

        # Chunk first so we know how many section labels to request.
        chunks = self._chunk_plain_text(text)
        n_chunks = len([c for c in chunks if c.strip()])

        # LLM-enriched metadata: title, section labels, page header/footer
        enriched = self._generate_metadata_enrichment(
            text, source_uri, n_chunks,
            {"title": "", "context_sentence": ctx_sentence},
        )
        doc_ctx = {
            "title": enriched.get("document_title", ""),
            "context_sentence": ctx_sentence,
        }
        sections = enriched.get("sections", [])
        page_header_default = enriched.get(
            "page_header_default", doc_ctx["title"]
        )
        page_footer_default = enriched.get("page_footer_default", "")

        output_chunks = []
        for pos, chunk_text in enumerate(chunks):
            if not chunk_text.strip():
                continue

            # Detect tables in free text (HTML or markdown pipe tables)
            is_html_table = "<table" in chunk_text.lower()
            is_md_table = bool(
                re.match(r"^\|.*\|\n\|[-:| ]+\|", chunk_text)
            )
            is_table = is_html_table or is_md_table

            chunk_ctx = {
                "page_header": page_header_default,
                "page_footer": page_footer_default,
                "section_header": (
                    sections[pos] if pos < len(sections) else ""
                ),
                "caption": "",
                "footnote": "",
                "page_number": str(pos + 1),
                "page_ids": [],
            }

            table_summary, related_qs = "", []
            if is_table:
                table_summary, related_qs = (
                    self._generate_table_enrichment(chunk_text)
                )
                if table_summary:
                    chunk_ctx["caption"] = table_summary

            output_chunks.append({
                "chunk_id": f"{doc_id}_{pos}",
                "chunk_position": pos,
                "chunk_to_retrieve": chunk_text,
                "chunk_to_embed": self._build_chunk_to_embed(
                    chunk_text, doc_ctx, chunk_ctx,
                    is_table=is_table,
                    table_summary=table_summary,
                    related_questions=related_qs,
                ),
                "metadata": {},
                "pages": [],
            })

        return {
            "document": {
                "contents": output_chunks,
                "pages": [],
                "source_uri": source_uri,
            },
            "error_status": [],
        }

    def _handle_row(self, row):
        parsed_json = row.get("parsed_doc_json", "")
        source_uri = row.get("source_uri") or ""
        if isinstance(source_uri, float) and pd.isna(source_uri):
            source_uri = ""
        doc_id = str(uuid.uuid4())

        elements, pages, _meta, is_plain = self._parse_input(parsed_json)

        self._llm_errors = []
        if is_plain:
            out = self._handle_plain_text(elements, source_uri, doc_id)
        else:
            out = self._handle_structured(elements, pages, source_uri, doc_id)
        # Surface LLM enrichment failures instead of silently returning blanks.
        out["error_status"] = out.get("error_status", []) + self._llm_errors
        return json.dumps(out, ensure_ascii=False)

    def predict(self, context, model_input, params=None):
        # Refresh token on every scoring call for long-lived replicas.
        self.client.api_key = self._get_token()

        if isinstance(model_input, pd.DataFrame):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)

        output = []
        for row in rows:
            try:
                output.append(self._handle_row(row))
            except Exception as exc:
                self.log.exception("Row failed")
                src = row.get("source_uri", "") if isinstance(row, dict) else ""
                output.append(json.dumps({
                    "document": {
                        "contents": [], "pages": [],
                        "source_uri": src,
                    },
                    "error_status": [f"processing_error: {exc}"],
                }, ensure_ascii=False))

        return pd.DataFrame({"response": output})

print(f"AiPrepSearchModel defined (chunking_model={CHUNKING_MODEL})")

In [0]:
# Build a sample ai_parse_document 2.0 envelope for the input example.
_SAMPLE_ENVELOPE = json.dumps({
    "document": {
        "pages": [{"id": 0, "image_uri": "sample.pdf"}],
        "elements": [
            {"id": 0, "type": "title", "content": "Sample Report",
             "confidence": 0.95,
             "bbox": [{"coord": [100, 50, 500, 80], "page_id": 0}],
             "description": None},
            {"id": 1, "type": "text",
             "content": "This is a sample document for testing.",
             "confidence": 0.90,
             "bbox": [{"coord": [100, 100, 500, 200], "page_id": 0}],
             "description": None},
        ],
    },
    "error_status": [],
    "metadata": {
        "id": "sample-001", "version": "2.0",
        "file_metadata": {
            "file_path": "sample.pdf", "file_name": "sample.pdf",
            "file_size": 1024,
            "file_modification_time": "1970-01-01T00:00:00Z",
        },
    },
})

signature = ModelSignature(
    inputs=Schema([
        ColSpec(DataType.string, "parsed_doc_json", required=True),
        ColSpec(DataType.string, "source_uri", required=False),
    ]),
    outputs=Schema([ColSpec(DataType.string, "response", required=True)]),
)

input_example = pd.DataFrame([{
    "parsed_doc_json": _SAMPLE_ENVELOPE,
    "source_uri": "/Volumes/catalog/schema/volume/sample.pdf",
}])

with mlflow.start_run(run_name="ai_prep_search_fmapi") as run:
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=AiPrepSearchModel(),
        signature=signature,
        input_example=input_example,
        resources=[
            DatabricksServingEndpoint(endpoint_name=CHUNKING_MODEL),
        ],
        pip_requirements=[
            "--extra-index-url=https://pypi.org/simple",
            "mlflow>=3.6.0",
            "openai>=1.37.0,<4.0.0",
            "pandas>=2.2.0",
            "numpy>=1.26.0,<3.0.0",
            "databricks-sdk>=0.62.0",
            "cloudpickle>=3.1.0",
        ],
        metadata={
            "chunking_model": CHUNKING_MODEL,
            "output_schema": "ai_prep_search/1.0",
        },
    )
    logged_run_id = run.info.run_id
    logged_model_uri = model_info.model_uri

# Register and alias
client = MlflowClient(registry_uri="databricks-uc")
try:
    client.get_registered_model(REGISTERED_MODEL_NAME)
except Exception:
    client.create_registered_model(REGISTERED_MODEL_NAME)

new_version = client.create_model_version(
    name=REGISTERED_MODEL_NAME,
    source=logged_model_uri,
    run_id=logged_run_id,
)
latest_model_version = new_version.version
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME, alias="Production",
    version=latest_model_version,
)
print({
    "run_id": logged_run_id,
    "model_uri": f"models:/{REGISTERED_MODEL_NAME}/{latest_model_version}",
    "alias": "Production",
})

In [0]:
# Create or update a CPU serving endpoint. No GPU needed — all inference
# is done via remote API calls to the foundation model.
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

workspace = WorkspaceClient()
served_entity = ServedEntityInput(
    name=ENDPOINT_NAME,
    entity_name=REGISTERED_MODEL_NAME,
    entity_version=latest_model_version,
    workload_size="Small",
    scale_to_zero_enabled=False,
)

try:
    workspace.serving_endpoints.get(name=ENDPOINT_NAME)
    workspace.serving_endpoints.update_config(
        name=ENDPOINT_NAME, served_entities=[served_entity]
    )
    action = "updated"
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(name=ENDPOINT_NAME, served_entities=[served_entity]),
    )
    action = "created"

# Poll until ready
deadline = time.time() + 60 * 60
while time.time() < deadline:
    endpoint = workspace.serving_endpoints.get(name=ENDPOINT_NAME)
    ready = str(getattr(endpoint.state, "ready", None))
    updating = str(getattr(endpoint.state, "config_update", None))
    print({"ready": ready, "config_update": updating})
    if "READY" in ready and (
        "NOT_UPDATING" in updating or updating in {"None", "READY"}
    ):
        break
    if "FAILED" in updating:
        raise RuntimeError(f"Endpoint deployment failed: {endpoint.state}")
    time.sleep(30)
else:
    raise TimeoutError(f"Timed out waiting for {ENDPOINT_NAME}")

print({"action": action, "endpoint": ENDPOINT_NAME,
       "version": latest_model_version})

## Smoke Tests

1. **Chained pipeline** — create a sample PDF from scratch, parse it with `ai-parse-document-fmapi`, then chunk/enrich with `ai-prep-search-fmapi`
2. **Free text** — run a sample free-text row directly through `ai-prep-search-fmapi`

In [0]:
# ── Test 1: Chained pipeline — PDF → ai-parse-document-fmapi → ai-prep-search-fmapi ──
import base64, io, json
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()

# Create a sample PDF from scratch (no volume dependency)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 11))
ax.set_title("Q3 2026 Financial Report", fontsize=22, fontweight="bold", pad=30)
ax.text(0.1, 0.85, "Executive Summary", fontsize=14, fontweight="bold")
ax.text(0.1, 0.75,
        "Revenue grew 20% year-over-year to $4.2 billion,\n"
        "driven by strong cloud adoption across enterprise\n"
        "customers. Operating margins improved to 32%.\n\n"
        "Customer retention remained above 95% with net\n"
        "revenue retention reaching 130%.",
        fontsize=11, va="top")
ax.text(0.1, 0.45, "Revenue Breakdown", fontsize=14, fontweight="bold")
ax.text(0.1, 0.35,
        "Cloud:    $2,800M (+28% YoY)\n"
        "On-Prem:  $1,400M (+8%  YoY)",
        fontsize=11, va="top", family="monospace")
ax.axis("off")

pdf_buf = io.BytesIO()
fig.savefig(pdf_buf, format="pdf", bbox_inches="tight")
plt.close(fig)
pdf_bytes = pdf_buf.getvalue()
pdf_b64 = base64.b64encode(pdf_bytes).decode()
print(f"Created sample PDF: {len(pdf_bytes):,} bytes")

# Step 1: Parse with ai-parse-document-fmapi
parse_resp = ws.serving_endpoints.query(
    name="ai-parse-document-fmapi",
    dataframe_records=[{
        "file_b64":               pdf_b64,
        "mime_type":              "application/pdf",
        "file_path":              "sample_report.pdf",
        "file_name":              "sample_report.pdf",
        "file_size":              len(pdf_bytes),
        "file_modification_time": "2026-01-01T00:00:00Z",
        "page_limit":             4,
        "page_index":             -1,
        "reformat":               True,
    }],
)
parsed_json_str = parse_resp.predictions[0]["response"]
parsed = json.loads(parsed_json_str)
n_el = len(parsed.get("document", {}).get("elements", []))
title = next((e["content"] for e in parsed["document"]["elements"]
              if e["type"] == "title"), "(none)")
print(f"\n✓ Step 1: ai-parse-document-fmapi → {n_el} elements (title: {title})")

# Step 2: Chain into ai-prep-search-fmapi
search_resp = ws.serving_endpoints.query(
    name="ai-prep-search-fmapi",
    dataframe_records=[{
        "parsed_doc_json": parsed_json_str,
        "source_uri":      "sample_report.pdf",
    }],
)
search_result = json.loads(search_resp.predictions[0]["response"])
chunks = search_result["document"]["contents"]
print(f"\n✓ Step 2: ai-prep-search-fmapi → {len(chunks)} search-ready chunks\n")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: pos={chunk['chunk_position']}, "
          f"retrieve={len(chunk['chunk_to_retrieve'])}ch, "
          f"embed={len(chunk['chunk_to_embed'])}ch")

print(f"\n✅ End-to-end: PDF ({len(pdf_bytes):,} bytes) → {n_el} elements → {len(chunks)} chunks")

# Register PDF as a temp view for the SQL ai_query test below
spark.createDataFrame([(pdf_bytes,)], ["content"]).createOrReplaceTempView("sample_pdf")

In [0]:
%sql
-- ── Test 1 (SQL): Chained pipeline via ai_query ────────────────────
-- Uses the `sample_pdf` temp view registered by the Python cell above.
WITH parsed AS (
  SELECT
    ai_query(
      'ai-parse-document-fmapi',
      named_struct(
        'file_b64',               base64(content),
        'mime_type',              'application/pdf',
        'file_path',              'sample_report.pdf',
        'file_name',              'sample_report.pdf',
        'file_size',              CAST(length(content) AS BIGINT),
        'file_modification_time', '2026-01-01T00:00:00Z',
        'page_limit',             CAST(4 AS BIGINT),
        'page_index',             CAST(-1 AS BIGINT),
        'reformat',               true
      ),
      returnType => 'STRUCT<response: STRING>'
    ).response AS parsed_json
  FROM sample_pdf
),
search AS (
  SELECT
    ai_query(
      'ai-prep-search-fmapi',
      named_struct(
        'parsed_doc_json', parsed_json,
        'source_uri',      'sample_report.pdf'
      ),
      returnType => 'STRUCT<response: STRING>'
    ).response AS search_json
  FROM parsed
)
SELECT
  chunk.value:chunk_id::STRING          AS chunk_id,
  chunk.value:chunk_position::INT       AS chunk_position,
  chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
  chunk.value:chunk_to_embed::STRING    AS chunk_to_embed
FROM search,
  LATERAL variant_explode(parse_json(search_json):document.contents) AS chunk
ORDER BY 2

In [0]:
# ── Test 2: Free text through ai-prep-search-fmapi ──────────────────
import json
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()

free_text = (
    "# Cloud Computing Overview\n\n"
    "Cloud computing delivers on-demand computing resources over the\n"
    "internet. It enables scalable infrastructure without upfront\n"
    "hardware investment.\n\n"
    "## Key Benefits\n\n"
    "Organizations benefit from elasticity, pay-per-use pricing, and\n"
    "managed services. Major providers include AWS, Azure, and Google\n"
    "Cloud.\n\n"
    "<table><tr><th>Provider</th><th>Market Share</th></tr>"
    "<tr><td>AWS</td><td>32%</td></tr>"
    "<tr><td>Azure</td><td>23%</td></tr>"
    "<tr><td>Google Cloud</td><td>11%</td></tr></table>\n\n"
    "## Security Considerations\n\n"
    "Data encryption, identity management, and compliance certifications\n"
    "are critical for enterprise cloud adoption."
)

resp = ws.serving_endpoints.query(
    name="ai-prep-search-fmapi",
    dataframe_records=[{
        "parsed_doc_json": free_text,
        "source_uri":      "cloud_overview.txt",
    }],
)
result = json.loads(resp.predictions[0]["response"])
chunks = result["document"]["contents"]
print(f"✓ Free text → {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    embed = chunk["chunk_to_embed"]
    print(f"--- Chunk {i} (pos={chunk['chunk_position']}) ---")
    print(f"retrieve: {chunk['chunk_to_retrieve'][:80]}...")
    for line in embed.split("\n")[:6]:
        print(f"  {line}")
    if len(embed.split("\n")) > 6:
        print("  ...")
    print()

print(f"✅ Free text: {len(chunks)} enriched chunks (title inferred, sections labeled, tables enriched)")

# Register free text as a temp view for the SQL ai_query test below
spark.createDataFrame([(free_text, "cloud_overview.txt")], ["text", "source_uri"]).createOrReplaceTempView("free_text_sample")

In [0]:
%sql
-- ── Test 2 (SQL): Free text via ai_query ────────────────────────────
-- Uses the `free_text_sample` temp view registered by the Python cell above.
WITH search AS (
  SELECT
    ai_query(
      'ai-prep-search-fmapi',
      named_struct(
        'parsed_doc_json', text,
        'source_uri',      source_uri
      ),
      returnType => 'STRUCT<response: STRING>'
    ).response AS search_json
  FROM free_text_sample
)
SELECT
  chunk.value:chunk_id::STRING          AS chunk_id,
  chunk.value:chunk_position::INT       AS chunk_position,
  chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
  chunk.value:chunk_to_embed::STRING    AS chunk_to_embed
FROM search,
  LATERAL variant_explode(parse_json(search_json):document.contents) AS chunk
ORDER BY 2